In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.config('spark.driver.host','localhost').appName('07').getOrCreate()

In [3]:
herb = spark.read.csv('C:\\datas\\3rd_week_data\\herbivore.csv',inferSchema=True,header=True)
carn = spark.read.csv('C:\\datas\\3rd_week_data\\carnivore.csv',inferSchema=True,header=True)

In [4]:
herb.printSchema()

root
 |-- name: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- eats_plants: string (nullable = true)



In [5]:
carn.printSchema()

root
 |-- name: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- eats_meat: string (nullable = true)



In [6]:
herb.show()

+------+---+-----------+
|  name| id|eats_plants|
+------+---+-----------+
| koala|  1|        yes|
|  deer|  2|        yes|
|   pig|  3|        yes|
|rabbit|  4|        yes|
| human|  5|        yes|
+------+---+-----------+



In [7]:
carn.show()

+-----+---+---------+
| name| id|eats_meat|
+-----+---+---------+
|shark|  6|      yes|
|  pig|  3|      yes|
| lion|  8|      yes|
|tiger|  9|      yes|
|human|  5|      yes|
+-----+---+---------+



## 1. `union`

- `DataFrame1.union(DataFrame2)`: DataFrame1과 2가 **상하로 결합**
- 일반적으로, 결합하는 두 DataFrame column의 **값과 순서가 동일한 경우 사용**

In [8]:
herb.union(carn).show()

+------+---+-----------+
|  name| id|eats_plants|
+------+---+-----------+
| koala|  1|        yes|
|  deer|  2|        yes|
|   pig|  3|        yes|
|rabbit|  4|        yes|
| human|  5|        yes|
| shark|  6|        yes|
|   pig|  3|        yes|
|  lion|  8|        yes|
| tiger|  9|        yes|
| human|  5|        yes|
+------+---+-----------+



- `df1.union(df2)`: `df2`의 column은 무시되고, `df1`의 column이 유지됨

In [10]:
# carn의 name과 id만 추출하여, herb와 결합
herb.union(carn.select('name','id'))

AnalysisException: [NUM_COLUMNS_MISMATCH] UNION can only be performed on inputs with the same number of columns, but the first input has 3 columns and the second input has 2 columns.;
'Union false, false
:- Relation [name#17,id#18,eats_plants#19] csv
+- Project [name#40, id#41]
   +- Relation [name#40,id#41,eats_meat#42] csv


- 결합하는 두 `DataFrame`의 column수가 다른 경우, 오류 발생

In [14]:
herb.select('id','name').union(carn.select('name','id')).show()

+-----+------+
|   id|  name|
+-----+------+
|    1| koala|
|    2|  deer|
|    3|   pig|
|    4|rabbit|
|    5| human|
|shark|     6|
|  pig|     3|
| lion|     8|
|tiger|     9|
|human|     5|
+-----+------+



- 결합 시, 동일한 이름을 갖는 `column` 또는 **type** 등을 고려하지 않고, 상하로 결합함
- 완전히 **column의 구성이 동일한 DataFrame 간의 결합**에서 사용됨

## 2. `join`

- `DataFrame1.join(DataFrame2, 조건, 결합방식)`
> - `조건`: **기준열**(복수로 선택 가능) 또는 **조건식**
> - `결합방식`: `how`를 통해서 결합방식을 지정함

- `how = 'inner'`: 두 DataFrame에 모두 존재하는 경우만 포함 (교집합)

In [16]:
# herb와 carn을 name과 id 기준으로 결합하되, 
# herb와 carn 모두 존재하는 name과 id에 데이터에 대해서만 포함시킴
herb.join(carn,['name','id'],how='inner').show()

+-----+---+-----------+---------+
| name| id|eats_plants|eats_meat|
+-----+---+-----------+---------+
|  pig|  3|        yes|      yes|
|human|  5|        yes|      yes|
+-----+---+-----------+---------+



- `how = 'left'`: 좌측 DataFrame 값을 기준으로 결합 (`df1.join(df2,...)`일 때, `df1`이 좌측 `DataFrame`)
- `left DataFrame`이 출력된 후, `right DataFrame`의 `Column`이 추가됨 (신규 Column에 값이 없는 경우, NULL)

In [18]:
# herb DataFrame의 name과 id를 기준으로 carn에 존재하는 열의 값을 추가함
herb.join(carn,['name','id'],how='left').show()

+------+---+-----------+---------+
|  name| id|eats_plants|eats_meat|
+------+---+-----------+---------+
| koala|  1|        yes|     NULL|
|  deer|  2|        yes|     NULL|
|   pig|  3|        yes|      yes|
|rabbit|  4|        yes|     NULL|
| human|  5|        yes|      yes|
+------+---+-----------+---------+



In [19]:
herb.join(carn,['id'],how='left').show()
# join의 기준열을 'id'로만 설정한 경우, id열만 공통으로 사용하며,
# carn의 name 열은 새로운 열로 추가됨

+---+------+-----------+-----+---------+
| id|  name|eats_plants| name|eats_meat|
+---+------+-----------+-----+---------+
|  1| koala|        yes| NULL|     NULL|
|  2|  deer|        yes| NULL|     NULL|
|  3|   pig|        yes|  pig|      yes|
|  4|rabbit|        yes| NULL|     NULL|
|  5| human|        yes|human|      yes|
+---+------+-----------+-----+---------+



- `how = 'right'`

In [20]:
herb.join(carn,['name','id'],how='right').show()

+-----+---+-----------+---------+
| name| id|eats_plants|eats_meat|
+-----+---+-----------+---------+
|shark|  6|       NULL|      yes|
|  pig|  3|        yes|      yes|
| lion|  8|       NULL|      yes|
|tiger|  9|       NULL|      yes|
|human|  5|        yes|      yes|
+-----+---+-----------+---------+



In [22]:
carn.join(herb,['name','id'],how='left').show()

+-----+---+---------+-----------+
| name| id|eats_meat|eats_plants|
+-----+---+---------+-----------+
|shark|  6|      yes|       NULL|
|  pig|  3|      yes|        yes|
| lion|  8|      yes|       NULL|
|tiger|  9|      yes|       NULL|
|human|  5|      yes|        yes|
+-----+---+---------+-----------+



- `df1.join(df2,..,how='left')`와 `df2.join(df1,..,how='right')`는 column의 순서를 제외하고는 결과가 동일함
- `left DataFrame`의 column이 결합 후에도 좌측에 나타남

- `how = 'outer'` 또는 `how = 'full'`: 기준열에 해당하는 모든 행을 반환, 결합하는 두 DataFrame에 모두 등장하는 행도 한번만 나타남 (합집합)
- 순수하게 두 DataFrame을 중복되지 않게 결합하고자 하는 경우 사용

In [25]:
herb.join(carn,['id','name'],how='outer').show()
# 결과는 첫번째 기준열 값 오름차순으로 정렬됨

+---+------+-----------+---------+
| id|  name|eats_plants|eats_meat|
+---+------+-----------+---------+
|  1| koala|        yes|     NULL|
|  2|  deer|        yes|     NULL|
|  3|   pig|        yes|      yes|
|  4|rabbit|        yes|     NULL|
|  5| human|        yes|      yes|
|  6| shark|       NULL|      yes|
|  8|  lion|       NULL|      yes|
|  9| tiger|       NULL|      yes|
+---+------+-----------+---------+



- 만약, 기준이 되는 column의 이름이 다를 경우, 조건식을 사용할 수도 있음
- 기준열을 설정하는 경우, 기준열을 결합 후, 공통으로 사용되나
- 조건식을 사용하는 경우, 기준열이 지정되지 않았으므로, 모든 열을 반환함

In [27]:
herb.join(carn,herb.id==carn.id,how='left').show()

+------+---+-----------+-----+----+---------+
|  name| id|eats_plants| name|  id|eats_meat|
+------+---+-----------+-----+----+---------+
| koala|  1|        yes| NULL|NULL|     NULL|
|  deer|  2|        yes| NULL|NULL|     NULL|
|   pig|  3|        yes|  pig|   3|      yes|
|rabbit|  4|        yes| NULL|NULL|     NULL|
| human|  5|        yes|human|   5|      yes|
+------+---+-----------+-----+----+---------+



## **연습문제**

In [30]:
course = spark.read.csv('C:\\datas\\3rd_week_data\\course_offerings.csv',header=True,inferSchema=True)
instructor = spark.read.csv('C:\\datas\\3rd_week_data\\instructors.csv',header=True,inferSchema=True)
section = spark.read.csv('C:\\datas\\3rd_week_data\\sections.csv',header=True,inferSchema=True)
teaching = spark.read.csv('C:\\datas\\3rd_week_data\\teachings.csv',header=True,inferSchema=True)

In [31]:
course.printSchema()
instructor.printSchema()
section.printSchema()
teaching.printSchema()

root
 |-- uuid: string (nullable = true)
 |-- course_uuid: string (nullable = true)
 |-- term_code: integer (nullable = true)
 |-- name: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)

root
 |-- uuid: string (nullable = true)
 |-- course_offering_uuid: string (nullable = true)
 |-- section_type: string (nullable = true)
 |-- number: integer (nullable = true)
 |-- room_uuid: string (nullable = true)
 |-- schedule_uuid: string (nullable = true)

root
 |-- instructor_id: integer (nullable = true)
 |-- section_uuid: string (nullable = true)



- `instructor.id`는 `teaching.instructor_id`와 동일
- `teaching.section_uuid`는 `section.uuid`와 동일
- `section.course_offering_uuid`는 `course.uuid`와 동일

#### **[문제]** 각 강사(`instructor`)에 대해 `instruct name, course_offer name, course_offer uuid`를 결합하여 보이시오.

#### **step 1.** `instructor`와 `teaching`의 결합
- `instructor`의 `id`열과 `teaching`의 `instructor_id`를 기준으로 결합

In [38]:
step1 = instructor.join(teaching,instructor.id==teaching.instructor_id,how='left')
step1.show(5)
# 한 명의 강사가 하나의 강의만 하는 것은 아니므로, 동일한 강사의 이름과 강사 id가 반환됨

+------+----------------+-------------+--------------------+
|    id|            name|instructor_id|        section_uuid|
+------+----------------+-------------+--------------------+
|761703|JOHN ARCHAMBAULT|       761703|45adf63c-48c9-365...|
|761703|JOHN ARCHAMBAULT|       761703|c6280e23-5e43-385...|
|761703|JOHN ARCHAMBAULT|       761703|9395dc21-15d1-3fa...|
|761703|JOHN ARCHAMBAULT|       761703|ca1c841f-41d5-329...|
|761703|JOHN ARCHAMBAULT|       761703|51c4dc00-1fc7-3c7...|
+------+----------------+-------------+--------------------+
only showing top 5 rows



#### **step 2.** **step1**의 결과와 `section`의 결합
- `step1`의 `section__uuid`와 `section`의 `uuid`를 기준으로 결합

In [40]:
step2 = step1.join(section,step1.section_uuid == section.uuid,how='left')
step2.show(5)

+-------+------------------+-------------+--------------------+--------------------+--------------------+------------+------+--------------------+--------------------+
|     id|              name|instructor_id|        section_uuid|                uuid|course_offering_uuid|section_type|number|           room_uuid|       schedule_uuid|
+-------+------------------+-------------+--------------------+--------------------+--------------------+------------+------+--------------------+--------------------+
| 590599|    RICHARD STRAUB|       590599|3ee98d8d-6b8e-34d...|3ee98d8d-6b8e-34d...|985a243e-86f7-308...|         LAB|   309|91002e7b-2635-3a5...|019a6bc9-90f4-3ee...|
| 590599|    RICHARD STRAUB|       590599|c2dce660-bf77-381...|c2dce660-bf77-381...|aeef8e65-6b2c-3c7...|         FLD|    12|                null|f2d66a4d-0c08-3b4...|
|1600463|KRISTIN KLARKOWSKI|      1600463|f1d3f955-21cc-35f...|f1d3f955-21cc-35f...|d2f83f7f-7283-3e2...|         IND|    41|                null|f2d66a4d-0c08-

#### **step 3.** **step2**의 결과와 course의 결합
- `step2`의 `course_offering_uuid`와 `course`의 `uuid`를 기준으로 결합

In [45]:
step3 = step2.join(course,step2.course_offering_uuid==course.uuid,how='left')
step3.show(5)

+-------+------------------+-------------+--------------------+--------------------+--------------------+------------+------+--------------------+--------------------+--------------------+--------------------+---------+--------------------+
|     id|              name|instructor_id|        section_uuid|                uuid|course_offering_uuid|section_type|number|           room_uuid|       schedule_uuid|                uuid|         course_uuid|term_code|                name|
+-------+------------------+-------------+--------------------+--------------------+--------------------+------------+------+--------------------+--------------------+--------------------+--------------------+---------+--------------------+
| 590599|    RICHARD STRAUB|       590599|3ee98d8d-6b8e-34d...|3ee98d8d-6b8e-34d...|985a243e-86f7-308...|         LAB|   309|91002e7b-2635-3a5...|019a6bc9-90f4-3ee...|985a243e-86f7-308...|d6d48ab8-f992-32f...|     1112|Intro to Engineer...|
| 590599|    RICHARD STRAUB|       5

#### **step 4.** **step3**의 결과에서 원하는 column만 추출

- `step3`에 `name`이라는 이름의 column이 2개 존재하므로, `step3`에서 `name`을 추출할 수 없음
- 그러므로 `step2`와 `course`를 결합할 때, column이름을 변경하여 추출함

In [50]:
step3 = step2.join(course,step2.course_offering_uuid==course.uuid,how='left').\
select(step2.name.alias('instructor_name'),course.name.alias('course_name'),course.uuid.alias('course_id'))
step3.show(5)

+------------------+--------------------+--------------------+
|   instructor_name|         course_name|           course_id|
+------------------+--------------------+--------------------+
|    RICHARD STRAUB|Intro to Engineer...|985a243e-86f7-308...|
|    RICHARD STRAUB|Cooperative Educ ...|aeef8e65-6b2c-3c7...|
|KRISTIN KLARKOWSKI|Coord Intrnshp/Co...|d2f83f7f-7283-3e2...|
|  JOHN ARCHAMBAULT|Cooperative Educa...|ea3b717c-d66b-30d...|
|  JOHN ARCHAMBAULT|Cooperative Educa...|075da420-5f49-3dd...|
+------------------+--------------------+--------------------+
only showing top 5 rows

